In [1]:
#    “Copyright (C) 2024 Mississippi State University.
 
#    This program is free software: you can redistribute it and/or modify
#    it under the terms of the GNU General Public License as published by
#    the Free Software Foundation, either version 3 of the License, or
#    (at your option) any later version.
 
#    This program is distributed in the hope that it will be useful,
#    but WITHOUT ANY WARRANTY; without even the implied warranty of
#    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
#    GNU General Public License for more details.
 
#    You should have received a copy of the GNU General Public License
#    along with this program.  If not, see <https://www.gnu.org/licenses/>.
 
# To inquire about relicensing, accessing more training data, collaborating with the author, or any general inquiry about the software, please contact Mississippi State University’s Office of Technology Management at otm@msstate.edu, (662) 325-9263.”


# Dependency Libraries

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask


# INPUTS

shapefile_path = r"C:\Users\Say70\OneDrive - Mississippi State University\Desktop\UGA talk\6_10_2020\shapefiles\polygons.shp"

raster_folder = r"C:\Users\Say70\OneDrive - Mississippi State University\Desktop\UGA talk\6_10_2020\Vegetation_Indices_Output"

output_excel = r"C:\Users\Say70\OneDrive - Mississippi State University\Desktop\UGA talk\6_10_2020\Vegetation_Indices_Output\subplot_indices_multisheet.xlsx"


# LOAD SHAPEFILE

gdf = gpd.read_file(shapefile_path)

print("Shapefile loaded")
print("Polygons:", len(gdf))


# FIND ALL INDEX RASTERS

raster_files = [
    os.path.join(raster_folder, f)
    for f in os.listdir(raster_folder)
    if f.endswith(".tif")
]

print("Total indices found:", len(raster_files))


# CREATE EXCEL WRITER

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:

    for raster_path in raster_files:
        raster_name = os.path.splitext(os.path.basename(raster_path))[0]
        print(f"Processing: {raster_name}")

        results = []

        with rasterio.open(raster_path) as src:
            raster_crs = src.crs
            nodata = src.nodata

            # Reproject shapefile if needed
            gdf_local = gdf.copy()
            if gdf_local.crs != raster_crs:
                gdf_local = gdf_local.to_crs(raster_crs)

            for idx, row in gdf_local.iterrows():
                geom = [row.geometry]

                try:
                    out_image, _ = mask(src, geom, crop=True, filled=False)
                    band = out_image[0]

                    # Extract valid pixels
                    if np.ma.isMaskedArray(band):
                        vals = band.compressed()
                    else:
                        vals = band[np.isfinite(band)]

                    vals = vals[np.isfinite(vals)]

                    if nodata is not None:
                        vals = vals[vals != nodata]

                    if len(vals) == 0:
                        stats = {
                            "polygon_id": idx + 1,
                            "count": 0,
                            "mean": np.nan,
                            "median": np.nan,
                            "min": np.nan,
                            "max": np.nan,
                            "std": np.nan
                        }
                    else:
                        stats = {
                            "polygon_id": idx + 1,
                            "count": len(vals),
                            "mean": np.mean(vals),
                            "median": np.median(vals),
                            "min": np.min(vals),
                            "max": np.max(vals),
                            "std": np.std(vals)
                        }

                    # Add shapefile attributes
                    for col in gdf_local.columns:
                        if col != "geometry":
                            stats[col] = row[col]

                    results.append(stats)

                except Exception as e:
                    print(f"Error polygon {idx+1}: {e}")
                    results.append({
                        "polygon_id": idx + 1,
                        "count": 0,
                        "mean": np.nan,
                        "median": np.nan,
                        "min": np.nan,
                        "max": np.nan,
                        "std": np.nan
                    })

        # Convert to DataFrame
        df = pd.DataFrame(results)

        # Sheet names must be <= 31 chars
        sheet_name = raster_name[:31]

        df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Saved Excel file with multiple sheets:")
print(output_excel)

Shapefile loaded
Polygons: 240
Total indices found: 19
Processing: masked_chlgr
Processing: masked_chlre
Processing: masked_dvi
Processing: masked_evi
Processing: masked_evi2
Processing: masked_gndvi
Processing: masked_mari
Processing: masked_mcari
Processing: masked_mcaridiosavi
Processing: masked_msavi
Processing: masked_msr
Processing: masked_ndvi
Processing: masked_osavi
Processing: masked_pvi
Processing: masked_savi
Processing: masked_savi2
Processing: masked_sr
Processing: masked_tsavi
Processing: masked_wdri
Saved Excel file with multiple sheets:
C:\Users\Say70\OneDrive - Mississippi State University\Desktop\UGA talk\6_10_2020\Vegetation_Indices_Output\subplot_indices_multisheet.xlsx


In [3]:
import pandas as pd

# =========================================================
# INPUT FILE
# =========================================================
excel_path = r"C:\Users\Say70\OneDrive - Mississippi State University\Desktop\UGA talk\6_10_2020\Vegetation_Indices_Output\subplot_indices_multisheet.xlsx"

output_excel = r"C:\Users\Say70\OneDrive - Mississippi State University\Desktop\UGA talk\6_10_2020\Vegetation_Indices_Output\selected_locations_mean_indices.xlsx"
output_csv   = r"C:\Users\Say70\OneDrive - Mississippi State University\Desktop\UGA talk\6_10_2020\Vegetation_Indices_Output\selected_locations_mean_indices.csv"

# =========================================================
# LOCATIONS TO KEEP (Since we do not have yield data for all locations, we will only keep the ones we have)
# =========================================================
selected_locations = [
    13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,
    61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,
    109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,
    157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,
    205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228
]
selected_locations = set(selected_locations)

# =========================================================
# READ ALL SHEETS
# =========================================================
xls = pd.ExcelFile(excel_path)

combined_df = None

for sheet_name in xls.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet_name)

    print(f"Processing sheet: {sheet_name}")
    print("Columns:", df.columns.tolist())

    # -----------------------------------------------------
    # Make sure required columns exist
    # -----------------------------------------------------
    if "location" not in df.columns:
        print(f"Skipping {sheet_name}: no 'location' column found.")
        continue

    if "mean" not in df.columns:
        print(f"Skipping {sheet_name}: no 'mean' column found.")
        continue

    # -----------------------------------------------------
    # Keep only selected locations
    # -----------------------------------------------------
    df_filtered = df[df["location"].isin(selected_locations)].copy()

    # Keep only location + mean
    df_filtered = df_filtered[["location", "mean"]].copy()

    # -----------------------------------------------------
    # Rename mean column using clean sheet name
    # Remove "masked_" prefix if present
    # -----------------------------------------------------
    clean_name = sheet_name.replace("masked_", "")
    df_filtered.rename(columns={"mean": clean_name}, inplace=True)

    # Drop duplicate locations if any
    df_filtered = df_filtered.drop_duplicates(subset="location")

    # Merge into final table
    if combined_df is None:
        combined_df = df_filtered
    else:
        combined_df = pd.merge(combined_df, df_filtered, on="location", how="outer")

# =========================================================
# SORT BY LOCATION
# =========================================================
if combined_df is not None:
    combined_df = combined_df.sort_values("location").reset_index(drop=True)

    # Save CSV
    combined_df.to_csv(output_csv, index=False)
    print(f"\nSaved CSV:\n{output_csv}")

    # Save Excel
    try:
        combined_df.to_excel(output_excel, index=False)
        print(f"Saved Excel:\n{output_excel}")
    except Exception as e:
        print("\nCould not save Excel.")
        print("Reason:", e)
        print("CSV was saved successfully instead.")

    print("\nPreview:")
    print(combined_df.head())
else:
    print("No matching sheets/columns were found.")

Processing sheet: masked_chlgr
Columns: ['polygon_id', 'count', 'mean', 'median', 'min', 'max', 'std', 'location', 'nit_perc', 'variety', 'border', 'rep', 'plot_id']
Processing sheet: masked_chlre
Columns: ['polygon_id', 'count', 'mean', 'median', 'min', 'max', 'std', 'location', 'nit_perc', 'variety', 'border', 'rep', 'plot_id']
Processing sheet: masked_dvi
Columns: ['polygon_id', 'count', 'mean', 'median', 'min', 'max', 'std', 'location', 'nit_perc', 'variety', 'border', 'rep', 'plot_id']
Processing sheet: masked_evi
Columns: ['polygon_id', 'count', 'mean', 'median', 'min', 'max', 'std', 'location', 'nit_perc', 'variety', 'border', 'rep', 'plot_id']
Processing sheet: masked_evi2
Columns: ['polygon_id', 'count', 'mean', 'median', 'min', 'max', 'std', 'location', 'nit_perc', 'variety', 'border', 'rep', 'plot_id']
Processing sheet: masked_gndvi
Columns: ['polygon_id', 'count', 'mean', 'median', 'min', 'max', 'std', 'location', 'nit_perc', 'variety', 'border', 'rep', 'plot_id']
Processin